In [1]:
import os
import sys
sys.path.append('/home/royhirsch/conformal/v3')

from ml_collections import config_dict
import logging
import pandas as pd
import pickle
import torch
import torch.nn as nn
import numpy as np
from sklearn.model_selection import train_test_split
from scipy.special import softmax
import torchcp
from data_config import get_config
from conformal_modules import APS, RAPS, Naive, SAPS, platt_scale

In [2]:
config = get_config('imagenet1k_resnet50') # cifar100_resnet56, imagenet1k_resnet50
config.par_calib = 0.5
config.num_calib = int(config.num_samples * config.par_calib)
config.plat_scale = True
config.seed = 4
config.estimation_method_name = 'max' # max / mean
config.randomized = True
config.no_zero_size_sets = True
config.alpha = 0.1

# Funcs

In [3]:
def get_estimated_score(scores, method, conf_score):
    if method == 'max':
        scores = conf_score.get_scores(scores, scores.argmax(1))
        return scores 
    
    elif method == 'mean':
        cal_pi = scores.argsort(1)[:, ::-1]
        cal_srt = np.take_along_axis(scores, cal_pi, axis=1).cumsum(axis=1)
        cal_softmax_correct_class = np.take_along_axis(cal_srt, cal_pi.argsort(axis=1), axis=1)
        return (cal_softmax_correct_class * scores).sum(1)
    else:
        raise ValueError('Unknown method')


with open(config.file_name, 'rb') as file:
    data = pickle.load(file)

labels_eval, labels_calib, scores_eval, scores_calib = train_test_split(
    data["labels"], data["preds"], test_size=config.par_calib, random_state=42
)
if config.plat_scale:
    t = platt_scale(scores_calib, labels_calib, max_iters=100, lr=0.01, epsilon=0.005)
else:
    t = 1.0

print(f"Temperature: {t}")
scores_calib = softmax(scores_calib / t, axis=1)
scores_eval = softmax(scores_eval / t, axis=1)

labels_tune, labels_calib, scores_tune, scores_calib = train_test_split(
    labels_calib, scores_calib, test_size=0.8, random_state=42
)

eval_data = {"labels": labels_eval, "scores": scores_eval}
calib_data = {"labels": labels_calib, "scores": scores_calib}
tune_data = {"labels": labels_tune, "scores": scores_tune}

print('Eval size:', len(labels_eval))
print('Calib size:', len(labels_calib))
print('Tune size:', len(labels_tune))

  1%|          | 1/100 [00:01<01:43,  1.05s/it]


Temperature: 0.6195021271705627
Eval size: 25000
Calib size: 20000
Tune size: 5000


In [4]:
conf_class = RAPS
# scan_range = np.insert(np.arange(0.05, 0.65, 0.05), 0, 0.02) # saps
scan_range = np.concatenate([np.asarray([0.001, 0.01]), np.arange(0.1, 0.55, 0.05)]) # raps

# Baselines

In [5]:

best_set_size = 1e5
ranking_weight_star = 0
for tmp_weight in scan_range:
    conf_score = conf_class(tmp_weight, randomized=True, no_zero_size_sets=config.no_zero_size_sets, seed=config.seed)
    cal_scores = conf_score.get_scores(tune_data['scores'], tune_data['labels'])
    n = len(scores_tune)
    baseline_qhat = np.quantile(
        cal_scores, np.ceil((n + 1) * (1 - config.alpha)) / n, interpolation="higher"
    )
    prediction_sets = conf_score.get_sets(tune_data['scores'], baseline_qhat)
    acc = prediction_sets[np.arange(prediction_sets.shape[0]), tune_data['labels']].mean()
    average_size = prediction_sets.sum(1).mean()
    if average_size < best_set_size:
        print('size :', average_size, 'acc :', acc)
        ranking_weight_star = tmp_weight
        best_set_size = average_size

print('Best weight:', ranking_weight_star)
calib_data = {'labels': labels_calib, 'scores': scores_calib}
n = len(calib_data["labels"])

conf_score = conf_class(ranking_weight_star, randomized=True, no_zero_size_sets=config.no_zero_size_sets, seed=config.seed)
cal_scores = conf_score.get_scores(calib_data["scores"], calib_data["labels"])
baseline_qhat = np.quantile(
    cal_scores, np.ceil((n + 1) * (1 - config.alpha)) / n, interpolation="higher"
)
prediction_sets = conf_score.get_sets(eval_data["scores"], baseline_qhat)

empirical_coverage = prediction_sets[
    np.arange(prediction_sets.shape[0]), eval_data["labels"]
].mean()

baseline_mets = {
    "size": prediction_sets.sum(1).mean(),
    "coverage": empirical_coverage,
}

print('Baseline mets:')
for key, value in baseline_mets.items():
    print(f"{key}: {value:.4f}")
print('Qhat: {:.4f}'.format(baseline_qhat))


size : 1.765 acc : 0.9004
size : 1.7644 acc : 0.9004
Best weight: 0.1
Baseline mets:
size: 1.8508
coverage: 0.8995
Qhat: 1.5650


# Ours method

In [6]:
best_set_size = 1e5
ranking_weight_star = 0
for tmp_weight in np.arange(0.05, 0.8, 0.05):
    conf_score = conf_class(tmp_weight, randomized=True, no_zero_size_sets=config.no_zero_size_sets, seed=config.seed)
    tune_scores_est = get_estimated_score(tune_data["scores"], config.estimation_method_name, conf_score)
    tune_scores_true = conf_score.get_scores(tune_data["scores"], tune_data["labels"])
    normalizer = tune_scores_true.max()
    eps = 1e-10
    residuales = (tune_scores_true - tune_scores_est) / (normalizer - tune_scores_est + eps)
    n = len(scores_tune)
    qhat = np.quantile(
        residuales, np.ceil((n + 1) * (1 - config.alpha)) / n, interpolation="higher"
    )
    prediction_sets = conf_score.get_sets(tune_data["scores"], tune_scores_est + qhat * (normalizer - tune_scores_est + eps))
    acc = prediction_sets[np.arange(prediction_sets.shape[0]), tune_data["labels"]].mean()
    average_size = prediction_sets.sum(1).mean()
    if average_size < best_set_size:
        print('size :', average_size, 'acc :', acc)
        ranking_weight_star = tmp_weight
        best_set_size = average_size

print('Best weight:', ranking_weight_star)
conf_score = conf_class(ranking_weight_star, randomized=True, no_zero_size_sets=config.no_zero_size_sets, seed=config.seed)
cal_scores_est = get_estimated_score(calib_data["scores"], config.estimation_method_name, conf_score)
cal_scores_true = conf_score.get_scores(calib_data["scores"], calib_data["labels"])
normalizer = cal_scores_true.max()
eps = 1e-10
residuales = (cal_scores_true - cal_scores_est) / (normalizer - cal_scores_est + eps)

print('Residuales: mean: {:.3f} std: {:.3f} min: {:.3f} max: {:.3f}'.format(residuales.mean(), residuales.std(), residuales.min(), residuales.max()))

qhat = np.quantile(
    residuales, np.ceil((n + 1) * (1 - config.alpha)) / n, interpolation="higher"
)

print('Qhat: {:.4f}'.format(qhat))
val_scores_est = get_estimated_score(eval_data["scores"], config.estimation_method_name, conf_score)
val_scores_est_copy = val_scores_est
val_scores_true = conf_score.get_scores(eval_data["scores"], eval_data["labels"])
# val_scores_est = np.zeros_like(val_scores_est)
val_scores_est = val_scores_est + qhat * (normalizer - val_scores_est + eps)
print('Par of scores above 1: {:.2f}'.format(np.sum(val_scores_est > 1) / len(val_scores_est)))
# val_scores_est[val_scores_est > 1] = baseline_qhat
prediction_sets = conf_score.get_sets(eval_data["scores"], val_scores_est)

empirical_coverage = prediction_sets[
    np.arange(prediction_sets.shape[0]), eval_data["labels"]
].mean()

ours_mets = {
    "size": prediction_sets.sum(1).mean(),
    "coverage": empirical_coverage,
}

print('Ours mets:')
for key, value in ours_mets.items():
    print(f"{key}: {value:.4f}")


size : 1.5798 acc : 0.9004
Best weight: 0.05
Residuales: mean: 0.002 std: 0.026 min: 0.000 max: 1.000
Qhat: 0.0010
Par of scores above 1: 0.98
Ours mets:
size: 1.8600
coverage: 0.8992


# Results

In [7]:
import pandas as pd

df = pd.DataFrame(columns=['Method', 'Size', 'Coverage'])
df.loc[0] = ['Baseline', baseline_mets['size'], baseline_mets['coverage']]
df.loc[1] = ['Ours', ours_mets['size'], ours_mets['coverage']]
print(df)

     Method     Size  Coverage
0  Baseline  1.85076   0.89952
1      Ours  1.86004   0.89916
